# Fusion Risk OS — Complete System Audit & Production Readiness Notebook

This notebook provides an automated validation and visualization harness for **Fusion Risk OS** (Unified Cyber-Fraud Intelligence Platform).
It checks system components, database tables, backend endpoints, ML model readiness, test suite integrity, and displays all generated audit reports.


## 1. System Inventory & Audit Reports Validation
Validate that all 12 required markdown audit reports exist and display their verification status.


In [5]:
import os
from IPython.display import display, Markdown

reports = [
    "SYSTEM_AUDIT_REPORT.md",
    "BACKEND_VALIDATION_REPORT.md",
    "DATABASE_HEALTH_REPORT.md",
    "NEO4J_HEALTH_REPORT.md",
    "ML_VALIDATION_REPORT.md",
    "PIPELINE_VALIDATION_REPORT.md",
    "FRONTEND_VALIDATION_REPORT.md",
    "VERCEL_DEPLOYMENT_REPORT.md",
    "PERFORMANCE_REPORT.md",
    "SECURITY_AUDIT_REPORT.md",
    "TEST_REPORT.md",
    "GLOBAL_HACKATHON_READINESS_REPORT.md"
]

print(f"Checking {len(reports)} audit reports...")
for r in reports:
    exists = os.path.exists(r)
    size = os.path.getsize(r) if exists else 0
    status = f"EXISTS ({size} bytes)" if exists else "MISSING"
    print(f"{r:<38} : {status}")


Checking 12 audit reports...
SYSTEM_AUDIT_REPORT.md                 : EXISTS (1509 bytes)
BACKEND_VALIDATION_REPORT.md           : EXISTS (955 bytes)
DATABASE_HEALTH_REPORT.md              : EXISTS (662 bytes)
NEO4J_HEALTH_REPORT.md                 : EXISTS (555 bytes)
ML_VALIDATION_REPORT.md                : EXISTS (667 bytes)
PIPELINE_VALIDATION_REPORT.md          : EXISTS (1109 bytes)
FRONTEND_VALIDATION_REPORT.md          : EXISTS (1015 bytes)
VERCEL_DEPLOYMENT_REPORT.md            : EXISTS (461 bytes)
PERFORMANCE_REPORT.md                  : EXISTS (441 bytes)
SECURITY_AUDIT_REPORT.md               : EXISTS (412 bytes)
TEST_REPORT.md                         : EXISTS (617 bytes)
GLOBAL_HACKATHON_READINESS_REPORT.md   : EXISTS (1306 bytes)


## 2. Automated Backend API Health Verification
Query the live FastAPI backend server (`http://localhost:8000`) for system readiness and platform status.


In [2]:
import urllib.request
import json

endpoints = [
    "http://localhost:8000/health/live",
    "http://localhost:8000/health/ready",
    "http://localhost:8000/platform/status",
    "http://localhost:8000/scenarios/list"
]

for url in endpoints:
    try:
        req = urllib.request.urlopen(url, timeout=3)
        data = json.loads(req.read().decode("utf-8"))
        print(f"[OK] {url:<40} -> Status: {req.status}")
        print(f"     Response: {json.dumps(data)[:100]}...")
    except Exception as e:
        print(f"[WARN] {url:<40} -> {e}")


[OK] http://localhost:8000/health/live        -> Status: 200
     Response: {"status": "live", "service": "fusion-risk-os"}...
[OK] http://localhost:8000/health/ready       -> Status: 200
     Response: {"status": "degraded", "service": "fusion-risk-os", "security_mode": "development", "dependencies": ...
[WARN] http://localhost:8000/platform/status    -> HTTP Error 401: Unauthorized
[WARN] http://localhost:8000/scenarios/list     -> HTTP Error 401: Unauthorized


## 3. Database Table & Store Inspection
Inspect the local SQLite store (`finspark.db`) and memory store collections.


In [3]:
import sqlite3
import os

db_path = "finspark.db"
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"Database {db_path} connected. Tables found: {len(tables)}")
    for (tbl,) in tables:
        cursor.execute(f'SELECT COUNT(*) FROM "{tbl}";')
        cnt = cursor.fetchone()[0]
        print(f"   - {tbl:<30} : {cnt} records")
    conn.close()
else:
    print(f"{db_path} not found directly in current dir, checking store module...")
    from api.store import list_all
    print(f"   - Customers count: {len(list_all('customers'))}")
    print(f"   - Transactions count: {len(list_all('transactions'))}")


Database finspark.db connected. Tables found: 6
   - store                          : 747 records
   - session_registry               : 0 records
   - trust_passports                : 0 records
   - trust_snapshots                : 0 records
   - trust_deltas                   : 0 records
   - trust_recovery_events          : 0 records


## 4. Machine Learning & Risk Engine Inference Test
Execute a live test transaction evaluation through `platform_pipeline`.


In [4]:
import asyncio
from api.core_platform.pipeline import platform_pipeline

test_txn = {
    "txn_id": "TXN_AUDIT_NOTEBOOK_001",
    "session_id": "SESS_NOTEBOOK_TEST",
    "event_type": "TRANSACTION_EVALUATION",
    "user_id": "usr_abc",
    "amount": 750000.0,
    "nameOrig": "ACC_ABC_123",
    "nameDest": "ACC_MULE_NEW",
    "cyber_compromise_in_window": True,
    "type": "TRANSFER"
}

async def run_eval():
    res = await platform_pipeline.process(test_txn, require_existing_session=False)
    print(f"Pre-transaction Pipeline Evaluation Result:")
    print(f"   - Decision: {res.decision['decision']}")
    print(f"   - Score: {res.inference['score']}/100")
    print(f"   - Reasons: {res.inference['reasons']}")
    print(f"   - Timings: {res.timings}")

asyncio.run(run_eval())


RuntimeError: asyncio.run() cannot be called from a running event loop

## 5. Gemini AI Copilot Integration Test
Test the Gemini AI Copilot chat handler with active candidate models.


In [6]:
from api.copilot_engine import chat_with_copilot, ChatRequest, ChatMessage

async def run_copilot_test():
    req = ChatRequest(messages=[ChatMessage(role="user", content="Summarize platform health and active threats")])
    res = await chat_with_copilot(req)
    print("Gemini Copilot Live Response:")
    print(res["response"])

asyncio.run(run_copilot_test())


c:\Users\motis\Downloads\fastapi\Unified-Cyber-Fraud-Intelligence-Platform\api\copilot_engine.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


RuntimeError: asyncio.run() cannot be called from a running event loop

## 6. Global Hackathon Readiness Report Display
Render the full **GLOBAL_HACKATHON_READINESS_REPORT.md** directly in markdown.


In [ ]:
if os.path.exists("GLOBAL_HACKATHON_READINESS_REPORT.md"):
    with open("GLOBAL_HACKATHON_READINESS_REPORT.md", "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print("GLOBAL_HACKATHON_READINESS_REPORT.md not found.")
